# 🧩 Query Decomposition — Break It Down to Search Better

## The Problem

Some questions are too complex for a single search.

```
❌ One search misses things:

   "Compare dropout and L2 regularization,
    and explain when to use each one"
        ↓
   One embedding can't capture all three parts
   (dropout concept + L2 concept + comparison logic)


✅ Decompose first:

   Sub-query 1: "What is dropout and how does it work?"
   Sub-query 2: "What is L2 regularization?"
   Sub-query 3: "When to use dropout vs L2 regularization?"
        ↓
   Search each separately → combine contexts
   → Much more complete answer!
```

## What You'll Learn

1. Why complex queries need decomposition
2. How to split a query into sub-queries
3. Search each sub-query, merge results
4. The full decomposition pipeline

In [1]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()  # reads ANTHROPIC_API_KEY from .env if present

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def llm(prompt, max_tokens=400):
    """Call Claude Haiku. Fast and cheap — perfect for RAG pipelines."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()

print("✅ Anthropic client ready!")
print("   Model: claude-haiku-4-5-20251001")
print("   Set ANTHROPIC_API_KEY in .env or environment.")


✅ Anthropic client ready!
   Model: claude-haiku-4-5-20251001
   Set ANTHROPIC_API_KEY in .env or environment.


In [2]:
# !pip install sentence-transformers

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Ready!")

Ready!


## 1. Knowledge Base

In [4]:
# Each document covers ONE specific concept
# This is realistic — real docs are focused, not encyclopedic

documents = [
    # Dropout
    "Dropout randomly deactivates neurons during training, forcing the network to learn redundant representations.",
    "Dropout rate is typically set between 0.2 and 0.5. Higher rates mean more regularization.",
    "Dropout is applied during training only. At inference, all neurons are active but outputs are scaled.",

    # L2 regularization
    "L2 regularization adds a penalty equal to the sum of squared weights to the loss function.",
    "L2 regularization discourages large weights, preventing the model from relying too heavily on any one feature.",
    "L2 regularization is also called weight decay and is a hyperparameter controlling regularization strength.",

    # When to use which
    "Dropout is most effective in large fully-connected layers and works well in computer vision and NLP.",
    "L2 regularization is preferred when interpretability matters, as it shrinks all weights smoothly.",
    "Both dropout and L2 regularization can be used together for stronger regularization.",

    # General overfitting
    "Overfitting happens when a model memorizes training data instead of learning general patterns.",
    "Regularization techniques like dropout and weight decay reduce overfitting in deep networks.",
]

doc_embeddings = model.encode(documents)
print(f"Indexed {len(documents)} documents")

Indexed 11 documents


## 2. Why One Query Isn't Enough

In [5]:
# Complex query — asks about 3 things at once
complex_query = "Compare dropout and L2 regularization, and explain when to use each"

# Search with the full complex query
q_emb = model.encode(complex_query)
scores = cosine_similarity([q_emb], doc_embeddings)[0]
top = np.argsort(scores)[::-1][:4]

print(f"Query: '{complex_query}'\n")
print("Top results (single query):")
for rank, idx in enumerate(top, 1):
    print(f"  {rank}. [{scores[idx]:.3f}] {documents[idx]}")

# What topics did we NOT get?
retrieved = set(top)
print(f"\nDocs about dropout:   {[i for i in range(3) if i in retrieved]}  (missing: {[i for i in range(3) if i not in retrieved]})")
print(f"Docs about L2:        {[i for i in range(3,6) if i in retrieved]}  (missing: {[i for i in range(3,6) if i not in retrieved]})")
print(f"Docs about when-to:   {[i for i in range(6,9) if i in retrieved]}  (missing: {[i for i in range(6,9) if i not in retrieved]})")

Query: 'Compare dropout and L2 regularization, and explain when to use each'

Top results (single query):
  1. [0.840] Both dropout and L2 regularization can be used together for stronger regularization.
  2. [0.658] Dropout rate is typically set between 0.2 and 0.5. Higher rates mean more regularization.
  3. [0.649] L2 regularization is preferred when interpretability matters, as it shrinks all weights smoothly.
  4. [0.589] L2 regularization adds a penalty equal to the sum of squared weights to the loss function.

Docs about dropout:   [1]  (missing: [0, 2])
Docs about L2:        [3]  (missing: [4, 5])
Docs about when-to:   [7, 8]  (missing: [6])


## 3. Decompose the Query

In [6]:
# Step 1: Break the complex query into focused sub-queries
# In production: ask an LLM to do this decomposition
# Here: we write them manually to understand the concept

sub_queries = [
    "What is dropout and how does it work?",           # Covers dropout
    "What is L2 regularization / weight decay?",       # Covers L2
    "When should I use dropout vs L2 regularization?", # Covers comparison
]

print("Original complex query:")
print(f"  '{complex_query}'\n")
print("Decomposed into sub-queries:")
for i, q in enumerate(sub_queries, 1):
    print(f"  {i}. {q}")

Original complex query:
  'Compare dropout and L2 regularization, and explain when to use each'

Decomposed into sub-queries:
  1. What is dropout and how does it work?
  2. What is L2 regularization / weight decay?
  3. When should I use dropout vs L2 regularization?


In [7]:
# Step 2: Search each sub-query separately

all_retrieved = {}  # doc_index → best score across all sub-queries

for sub_q in sub_queries:
    emb = model.encode(sub_q)
    scores = cosine_similarity([emb], doc_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:3]  # Top 3 per sub-query
    
    print(f"Sub-query: '{sub_q}'")
    for idx in top_indices:
        print(f"  [{scores[idx]:.3f}] {documents[idx][:65]}...")
        # Store the best score this doc received
        if idx not in all_retrieved or scores[idx] > all_retrieved[idx]:
            all_retrieved[idx] = float(scores[idx])
    print()

Sub-query: 'What is dropout and how does it work?'
  [0.626] Dropout rate is typically set between 0.2 and 0.5. Higher rates m...
  [0.623] Dropout is applied during training only. At inference, all neuron...
  [0.594] Dropout randomly deactivates neurons during training, forcing the...

Sub-query: 'What is L2 regularization / weight decay?'
  [0.916] L2 regularization is also called weight decay and is a hyperparam...
  [0.746] L2 regularization adds a penalty equal to the sum of squared weig...
  [0.707] L2 regularization is preferred when interpretability matters, as ...

Sub-query: 'When should I use dropout vs L2 regularization?'
  [0.842] Both dropout and L2 regularization can be used together for stron...
  [0.674] Dropout rate is typically set between 0.2 and 0.5. Higher rates m...
  [0.661] L2 regularization is preferred when interpretability matters, as ...



In [8]:
# Step 3: Merge results — sort all retrieved docs by best score

merged = sorted(all_retrieved.items(), key=lambda x: x[1], reverse=True)

print("Final merged results (all sub-queries combined):\n")
for rank, (idx, score) in enumerate(merged, 1):
    print(f"  {rank}. [{score:.3f}] {documents[idx]}")

print(f"\n✅ Retrieved {len(merged)} unique docs (vs 4 with a single query)")

Final merged results (all sub-queries combined):

  1. [0.916] L2 regularization is also called weight decay and is a hyperparameter controlling regularization strength.
  2. [0.842] Both dropout and L2 regularization can be used together for stronger regularization.
  3. [0.746] L2 regularization adds a penalty equal to the sum of squared weights to the loss function.
  4. [0.707] L2 regularization is preferred when interpretability matters, as it shrinks all weights smoothly.
  5. [0.674] Dropout rate is typically set between 0.2 and 0.5. Higher rates mean more regularization.
  6. [0.623] Dropout is applied during training only. At inference, all neurons are active but outputs are scaled.
  7. [0.594] Dropout randomly deactivates neurons during training, forcing the network to learn redundant representations.

✅ Retrieved 7 unique docs (vs 4 with a single query)


## 4. LLM-Based Decomposition

In production, you ask an LLM to do the decomposition. Here's the pattern:

In [9]:
# The prompt you'd send to the LLM
def decomposition_prompt(question):
    return f"""Break the following question into 2-4 simpler, focused sub-questions.
Each sub-question should be answerable on its own.
Output one sub-question per line, nothing else.

Question: {question}"""

print(decomposition_prompt(complex_query))

Break the following question into 2-4 simpler, focused sub-questions.
Each sub-question should be answerable on its own.
Output one sub-question per line, nothing else.

Question: Compare dropout and L2 regularization, and explain when to use each


In [10]:
# Decompose a complex question into sub-questions using Claude

def decompose_question(question, n=3):
    """Ask Claude to break a complex question into simpler sub-questions."""
    prompt = f"""Break the following question into {n} simpler sub-questions.
Return ONLY a numbered list, one sub-question per line.

Question: {question}
Sub-questions:"""
    response = llm(prompt, max_tokens=200)
    # Parse numbered lines into a list
    sub_qs = []
    for line in response.strip().split('\n'):
        line = line.strip()
        if line and line[0].isdigit():
            # Remove leading number and punctuation
            sub_qs.append(line.lstrip('0123456789.). ').strip())
    return sub_qs if sub_qs else [question]  # fallback


# Test it
question = "Compare dropout and L2 regularization for preventing overfitting"
sub_questions = decompose_question(question)

print(f"Original: {question}\n")
print("Decomposed into:")
for i, sq in enumerate(sub_questions, 1):
    print(f"  {i}. {sq}")


Original: Compare dropout and L2 regularization for preventing overfitting

Decomposed into:
  1. What is dropout and how does it work to prevent overfitting?
  2. What is L2 regularization and how does it work to prevent overfitting?
  3. What are the key differences and trade-offs between dropout and L2 regularization?


## 5. Full Pipeline

In [11]:
# Complete decomposition search in one flow

question = "Compare dropout and L2 regularization, and explain when to use each"

# Step 1: Decompose
sub_qs = decompose_question(question)
print(f"1. Decomposed into {len(sub_qs)} sub-queries")

# Step 2: Search each sub-query, collect results
collected = {}  # doc_index → score

for sq in sub_qs:
    emb = model.encode(sq)
    scores = cosine_similarity([emb], doc_embeddings)[0]
    for idx in np.argsort(scores)[::-1][:3]:
        if idx not in collected or scores[idx] > collected[idx]:
            collected[idx] = float(scores[idx])

print(f"2. Retrieved {len(collected)} unique documents")

# Step 3: Rank and return top results
final = sorted(collected.items(), key=lambda x: x[1], reverse=True)[:6]

print(f"3. Top {len(final)} results:\n")
for rank, (idx, score) in enumerate(final, 1):
    print(f"   {rank}. [{score:.3f}] {documents[idx]}")

# Compare against single query
single_emb = model.encode(question)
single_scores = cosine_similarity([single_emb], doc_embeddings)[0]
single_top = set(np.argsort(single_scores)[::-1][:4])
decomp_top = {idx for idx, _ in final}

print(f"\n📊 Single query top-4 doc indices:       {sorted(single_top)}")
print(f"   Decomposition top-{len(final)} doc indices:  {sorted(decomp_top)}")
print(f"   New docs found by decomposition: {sorted(decomp_top - single_top)}")

1. Decomposed into 3 sub-queries
2. Retrieved 6 unique documents
3. Top 6 results:

   1. [0.831] Both dropout and L2 regularization can be used together for stronger regularization.
   2. [0.800] L2 regularization is also called weight decay and is a hyperparameter controlling regularization strength.
   3. [0.795] L2 regularization adds a penalty equal to the sum of squared weights to the loss function.
   4. [0.742] Dropout rate is typically set between 0.2 and 0.5. Higher rates mean more regularization.
   5. [0.723] L2 regularization is preferred when interpretability matters, as it shrinks all weights smoothly.
   6. [0.631] Dropout is applied during training only. At inference, all neurons are active but outputs are scaled.

📊 Single query top-4 doc indices:       [1, 3, 7, 8]
   Decomposition top-6 doc indices:  [1, 2, 3, 5, 7, 8]
   New docs found by decomposition: [2, 5]


## Key Takeaways 🎯

| | Single Query | Decomposition |
|---|---|---|
| Best for | Simple, focused questions | Multi-part, complex questions |
| Coverage | One region of vector space | Multiple targeted regions |
| Cost | 1 embed call | N embed calls (1 per sub-query) |
| Needs LLM? | No | For best quality, yes |

**Rule of thumb:** If your question has the word "and", "compare", or "vs" — decompose it.

---
Next: `05_Routing_and_Filtering.ipynb` — route queries to the right subset of your data.